# CNN-LSTM Signal Peptide Classifier — Model Definition & Hyperparameter Optimisation

This notebook defines the full deep-learning pipeline for predicting signal peptides from protein sequences.



## 1. Imports & Device Configuration

We detect GPU availability here once and expose `device` as a module-level variable.
All models and tensors will be moved to this device during training and inference.

In [1]:
import gc
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import matthews_corrcoef
from torch.utils.data import DataLoader, Dataset

# Automatically use GPU if available, otherwise fall back to CPU.
# This single variable is referenced everywhere tensors are moved to hardware.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 2. One-Hot Encoding

### Why one-hot encoding?
Neural networks require numerical input. Each amino acid is represented as a
21-dimensional binary vector (20 standard amino acids + 1 padding token `X`).
A sequence of length 90 becomes a **90 × 21 matrix** — the direct input to the CNN.

### Why 90 residues?
Signal peptides are always located at the N-terminus and are typically 15–30 residues long.
Truncating at 90 residues preserves all biologically relevant positional information
while keeping the input size fixed and computationally manageable.

### Padding token `X`
Sequences shorter than 90 are padded with `X`, which encodes as an all-zero vector.
This lets the model explicitly distinguish real residues from padding.

In [2]:
def _increase_lenseq(seq: str, target: int = 90) -> str:
    """
    Pad a protein sequence with 'X' characters until it reaches `target` length.
    'X' encodes as an all-zero vector, so padding is informationally inert.
    """
    return seq + "X" * (target - len(seq))


def one_hot_encoding(sequence: str) -> np.ndarray:
    """
    Convert a fixed-length amino-acid sequence into a binary matrix.

    Alphabet (21 channels): A C D E F G H I K L M N P Q R S T V W Y X
    The last channel (X) captures both explicit padding and unknown residues.

    Returns
    -------
    np.ndarray  shape (len(sequence), 21), dtype float32
    """
    AA_ALPHABET = ["A", "C", "D", "E", "F", "G", "H", "I", "K", "L",
                   "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "X"]
    # Pre-build a lookup dict for O(1) index retrieval per residue
    aa_to_idx = {aa: i for i, aa in enumerate(AA_ALPHABET)}

    matrix = []
    for aa in sequence:
        one_hot = np.zeros(21, dtype=np.float32)
        idx = aa_to_idx.get(aa)   # returns None for truly unknown characters
        if idx is not None:
            one_hot[idx] = 1.0
        # Completely unknown characters remain all-zero — safe, neutral fallback
        matrix.append(one_hot)

    return np.array(matrix, dtype=np.float32)


def create_one_hot_sets(dataset: pd.DataFrame) -> list:
    """
    Iterate over the 5 cross-validation folds, standardise each sequence to
    90 residues, apply one-hot encoding, and return (X, y) arrays per fold.

    Parameters
    ----------
    dataset : pd.DataFrame
        Must contain columns: Sequence, Class ('Positive'/'Negative'), Set ('1'–'5').

    Returns
    -------
    list of 5 tuples (X, y):
        X : np.ndarray  float32  shape (n_samples, 90, 21)
        y : np.ndarray  float32  shape (n_samples,)   values 0 or 1
    """
    set_results = []

    for fold_id in ["1", "2", "3", "4", "5"]:
        subset = dataset.query(f"Set == '{fold_id}'")
        X_list, y_list = [], []

        for _, row in subset.iterrows():
            seq = row["Sequence"]

            # Truncate long sequences (keep N-terminus) or pad short ones
            seq = seq[:90] if len(seq) >= 90 else _increase_lenseq(seq)

            X_list.append(one_hot_encoding(seq))
            # Binary labels: 1 = signal peptide present, 0 = no signal peptide
            y_list.append(1 if row["Class"] == "Positive" else 0)

        set_results.append((
            np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.float32),
        ))

    return set_results

## 3. Model Architecture — Hybrid CNN-LSTM (`SP_NN`)

The model combines three complementary components:

```
Input  [B, 90, 21]
  │
  ▼
Conv1D  (kernel=17, 64 maps, padding='same')   ← local motif detection
  │  [B, 90, 64]
  ▼
LSTM   (stacked layers, hidden=128)            ← sequential context
  │  take last hidden state  [B, 128]
  ▼
BatchNorm
  │
  ▼
MLP    (Linear → ReLU → Dropout) × n_layers   ← classification
  │
  ▼
Sigmoid  → probability ∈ (0, 1)
```

**CNN kernel = 17:** Signal peptide hydrophobic cores are ~7–15 residues long.
A kernel of 17 comfortably spans this region, detecting hydrophobicity patterns
in a single convolution step.

**Last hidden state:** The LSTM reads the entire 90-residue sequence and
its final hidden state serves as a learned summary vector encoding all
sequential information seen up to the last position.

**Dynamic MLP head:** `hidden_sizes` is a list, so any depth and width
can be tested during hyperparameter search without code changes.

In [3]:
class SP_NN(nn.Module):
    """
    Hybrid CNN-LSTM neural network for binary signal-peptide classification.

    Parameters
    ----------
    input_size       : int   – one-hot dimension, always 21
    hidden_sizes     : list  – width of each MLP hidden layer (e.g. [256, 128, 64, 1024])
    lstm_hidden_size : int   – LSTM hidden-state dimension
    num_lstm_layers  : int   – number of stacked LSTM layers
    output_size      : int   – 1 for binary classification
    dropout_p        : float – dropout probability applied in MLP and inter-LSTM layers
    """

    def __init__(
        self,
        input_size: int,
        hidden_sizes: list,
        lstm_hidden_size: int,
        num_lstm_layers: int,
        output_size: int,
        dropout_p: float = 0.5,
    ):
        super().__init__()

        # ── Block 1: Convolutional feature extractor ────────────────────────
        # 64 feature maps capture diverse local sequence patterns simultaneously.
        # padding='same' preserves the sequence length (90 → 90) after convolution.
        self.cnn_out_channels = 64
        self.conv1 = nn.Conv1d(
            in_channels=input_size,       # 21 one-hot channels
            out_channels=self.cnn_out_channels,
            kernel_size=17,               # covers typical hydrophobic-core length
            padding="same",
        )

        # ── Block 2: LSTM sequential processor ─────────────────────────────
        # Dropout between stacked LSTM layers is only valid when num_layers > 1;
        # passing dropout > 0 to a single-layer LSTM raises a PyTorch UserWarning.
        lstm_dropout = dropout_p if num_lstm_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=self.cnn_out_channels,  # receives CNN feature maps
            hidden_size=lstm_hidden_size,
            num_layers=num_lstm_layers,
            batch_first=True,                  # input/output shape: [B, T, F]
            dropout=lstm_dropout,
        )

        # BatchNorm stabilises the LSTM output distribution before the classifier,
        # reducing internal covariate shift and speeding up convergence.
        self.bn = nn.BatchNorm1d(lstm_hidden_size)

        # ── Block 3: Dynamic MLP classification head ────────────────────────
        # Built from a list so architecture can vary without code changes.
        mlp_layers = []
        current_size = lstm_hidden_size
        for hidden_size in hidden_sizes:
            mlp_layers += [
                nn.Linear(current_size, hidden_size),
                nn.ReLU(),               # non-linearity for expressive power
                nn.Dropout(p=dropout_p), # regularisation against overfitting
            ]
            current_size = hidden_size
        # Final layer: project to scalar → Sigmoid squashes to (0, 1) probability
        mlp_layers += [nn.Linear(current_size, output_size), nn.Sigmoid()]
        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Input x: [Batch, Length=90, Channels=21]

        # Conv1d requires channel-first format: permute to [B, 21, 90]
        x = x.permute(0, 2, 1)
        x = self.conv1(x)           # → [B, 64, 90]  (same length, more features)

        # LSTM requires sequence-first format: permute back to [B, 90, 64]
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)       # → [B, 90, lstm_hidden_size]

        # Summarise the whole sequence with the LAST time-step hidden state.
        # This is equivalent to asking: "after reading all 90 positions, what
        # does the model think about this sequence overall?"
        out = out[:, -1, :]         # → [B, lstm_hidden_size]
        out = self.bn(out)          # normalise before classification head

        return self.mlp(out)        # → [B, 1]  probability of being a signal peptide

## 4. Custom Dataset — `SignalDataset`

### Lazy loading strategy
One-hot encoding a dataset of ~10,000 proteins at 90×21 floats produces arrays
large enough to strain RAM if converted to tensors all at once.

`SignalDataset` stores raw NumPy arrays and converts **only the requested sample**
to a tensor inside `__getitem__`. The `DataLoader` then constructs mini-batches
on the fly, keeping peak memory usage low.

In [4]:
class SignalDataset(Dataset):
    """
    Memory-efficient dataset wrapper using lazy tensor conversion.

    The class stores raw NumPy arrays and converts individual samples to
    PyTorch tensors only when the DataLoader requests them.  This prevents
    duplicating the entire dataset in memory as both NumPy arrays and tensors.
    """

    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Store as NumPy arrays — no tensor conversion here
        self.X = X
        self.y = y

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        x_out = torch.from_numpy(self.X[idx]).float()

        # Reshape label from scalar to [1] so it matches the model output shape [B, 1].
        # Without .view(1), BCELoss raises a shape mismatch error.
        y_out = torch.tensor(self.y[idx], dtype=torch.float32).view(1)

        return x_out, y_out

## 5. Training Loop — `train_val`

Key engineering decisions:

| Technique | Why |
|---|---|
| **Gradient clipping** (`max_norm=1.0`) | LSTMs are prone to exploding gradients. Capping the gradient norm prevents weight updates from blowing up and destabilising training. |
| **Early stopping** | Monitors validation MCC. If it doesn't improve for `patience` epochs, training stops — preventing overfitting and saving compute. |
| **Best-model tracking** | We save the weights of the best-scoring epoch, not the last. The returned `state_dict` is always the optimal checkpoint. |
| **MCC as stopping metric** | Accuracy is misleading on imbalanced datasets. MCC accounts for all four cells of the confusion matrix and is a fairer measure of binary classification quality. |

In [5]:
def train_val(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs: int,
    patience: int,
    scorer=matthews_corrcoef,
    init_best_score: float = -1.0,
    output_transform=lambda x: (x > 0.5).float(),
    verbose: bool = True,
) -> dict:
    """
    Training and validation loop with early stopping and gradient clipping.

    Parameters
    ----------
    model            : nn.Module
    train_loader     : DataLoader — shuffled training batches
    val_loader       : DataLoader — validation batches (no shuffle)
    optimizer        : e.g. Adam
    criterion        : loss function, e.g. BCELoss
    epochs           : maximum number of training passes
    patience         : stop if no improvement for this many consecutive epochs
    scorer           : metric function (default: MCC)
    init_best_score  : float — baseline score; -1 means any positive MCC is an improvement
    output_transform : converts raw model probabilities to class labels (default: threshold 0.5)
    verbose          : print per-epoch progress

    Returns
    -------
    dict — state_dict of the epoch with the highest validation score
    """
    best_val_score = init_best_score
    epochs_without_improvement = 0
    best_model_state_dict = None   # will hold the best checkpoint in RAM

    for epoch in range(epochs):

        # ── Training phase ────────────────────────────────────────────────────
        model.train()   # enables Dropout and updates BatchNorm running stats
        epoch_loss = 0.0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()            # clear gradients from previous step
            outputs = model(batch_X)         # forward pass
            loss = criterion(outputs, batch_y)
            loss.backward()                  # compute gradients

            # Gradient clipping: rescale gradient vector if its norm exceeds 1.0.
            # This is critical for LSTMs where gradients can grow exponentially
            # over long sequences.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()                 # update weights
            epoch_loss = loss.item()         # save last-batch loss for logging

        # ── Validation phase ──────────────────────────────────────────────────
        model.eval()    # disables Dropout; freezes BatchNorm running stats
        val_preds, val_labels = [], []

        with torch.no_grad():   # skip gradient computation — saves memory and time
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                preds = output_transform(model(batch_X))
                val_preds.extend(preds.cpu().numpy().flatten())
                val_labels.extend(batch_y.cpu().numpy().flatten())

        val_score = scorer(val_labels, val_preds)

        if verbose:
            print(f"Epoch [{epoch + 1}/{epochs}]  Loss: {epoch_loss:.4f}  Val MCC: {val_score:.4f}")

        # ── Early stopping logic ──────────────────────────────────────────────
        if val_score > best_val_score:
            best_val_score = val_score
            epochs_without_improvement = 0
            # Deep-copy weights into CPU memory so they survive further training
            best_model_state_dict = model.state_dict()
            if verbose:
                print(f"  ✓ Improved to {best_val_score:.4f} — checkpoint saved")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                if verbose:
                    print(f"  Early stopping triggered at epoch {epoch + 1}")
                break

    return best_model_state_dict

## 5b. Inference — `test`

Evaluates a trained model on a held-out set.

- `model.eval()` disables Dropout and freezes BatchNorm so results are deterministic.  
- `torch.no_grad()` skips the autograd graph — faster and uses less GPU memory.  
- Returns both the scalar MCC and the full prediction list so callers can do further error analysis.

In [6]:
def test(
    model,
    test_loader,
    scorer=matthews_corrcoef,
    output_transform=lambda x: (x > 0.5).float(),
):
    """
    Evaluate a trained (or JIT-traced) model on a test set.

    Parameters
    ----------
    model            : nn.Module or TorchScript module
    test_loader      : DataLoader
    scorer           : metric function (default: MCC)
    output_transform : probability → class label (default: threshold > 0.5)

    Returns
    -------
    score     : float — MCC over the entire test set
    all_preds : list  — binary prediction for every sample (0 or 1)
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            preds = output_transform(model(batch_X))
            # Move to CPU before converting to NumPy; sklearn metrics do not accept CUDA tensors
            all_preds.extend(preds.cpu().numpy().flatten())
            all_labels.extend(batch_y.cpu().numpy().flatten())

    score = scorer(all_labels, all_preds)
    return score, all_preds

## 6. Hyperparameter Optimisation — Ray Tune

### Why automated HPO?
The search space (LSTM depth/width, MLP topology, learning rate, dropout, batch size)
has too many interactions to tune manually.

### Strategy
- **Random search + ASHA scheduler:** 15 trials are sampled randomly; ASHA prunes
  trials that fall behind early, saving GPU time.
- **5-fold cross-validation per trial:** Each configuration is evaluated on all 5
  folds and the average MCC is reported. This prevents lucky splits from dominating.
- **Shared Ray Object Store:** The encoded dataset is stored once in Ray's shared
  memory and accessed by reference in each worker — avoids serialising 100+ MB per trial.
- **MLP topologies tested:**
  - *Funnel* (descending sizes) — compresses features progressively, stable but conservative
  - *Flexible* (shuffled sizes) — allows intermediate expansion, more expressive

### Result
The best configuration found across all trials is hardcoded into `benchmark_test.ipynb`
for the final production run.

In [7]:
def _generate_hidden_sizes(config: dict) -> list:
    """
    Ray Tune helper: sample a dynamic MLP-head architecture for one trial.

    Randomly chooses between two design philosophies:
    - Funnel  : sizes in descending order (e.g. 512 → 256 → 64)  — compressive
    - Flexible: sizes shuffled randomly  (e.g. 128 → 512 → 64)  — exploratory
    """
    pool = [1024, 512, 256, 128, 64, 32]
    num_layers = min(config["num_layers"], len(pool))
    dims = random.sample(pool, num_layers)   # sample without replacement

    if random.choice([True, False]):
        return sorted(dims, reverse=True)   # Funnel: biggest layer first
    else:
        random.shuffle(dims)
        return dims                         # Flexible: random order


def _tune_trainable(config: dict) -> None:
    """
    Ray Tune trainable function.
    Runs a full 5-fold cross-validation for `config` and reports average MCC.
    """
    import ray
    from ray import tune

    # Retrieve dataset from shared object store (avoids per-worker copy)
    all_data = ray.get(config["data_ref"])
    mcc_scores = []
    _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for i in range(5):
        # Rotating fold assignment: testing=i, validation=(i+4)%5, training=rest
        training_indices = [(i + 1) % 5, (i + 2) % 5, (i + 3) % 5]
        validation_index = (i + 4) % 5
        testing_index    = i

        x_train = np.concatenate([all_data[j][0] for j in training_indices])
        y_train = np.concatenate([all_data[j][1] for j in training_indices])
        x_val,  y_val  = all_data[validation_index]
        x_test, y_test = all_data[testing_index]

        train_loader = DataLoader(SignalDataset(x_train, y_train), batch_size=config["batch_size"], shuffle=True)
        val_loader   = DataLoader(SignalDataset(x_val,   y_val),   batch_size=config["batch_size"])
        test_loader  = DataLoader(SignalDataset(x_test,  y_test),  batch_size=config["batch_size"])

        model = SP_NN(
            input_size=21,
            hidden_sizes=config["hidden_sizes"],
            lstm_hidden_size=config["lstm_hidden_size"],
            num_lstm_layers=config["num_lstm_layers"],
            output_size=1,
            dropout_p=config["dropout"],
        ).to(_device)

        optimizer = optim.Adam(model.parameters(), lr=config["lr"])
        criterion = nn.BCELoss()

        best_state = train_val(model, train_loader, val_loader, optimizer, criterion,
                               epochs=100, patience=20, verbose=False)
        model.load_state_dict(best_state)

        mcc, _ = test(model, test_loader)   # unpack (score, preds) tuple
        mcc_scores.append(mcc)

    mean_mcc = float(np.mean(mcc_scores))
    # Ray minimises 'loss' by default, so report -mean_mcc to maximise MCC
    tune.report({"mcc": mean_mcc, "loss": -mean_mcc,
                 "hidden_layers_size": str(config["hidden_sizes"])})

In [8]:
# ── Install Ray Tune (run once) ───────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-U", "typing_extensions", "pydantic", "pydantic_core", "ray[tune]"])

0

In [9]:
import ray
from ray import tune

# ── 1. Initialise Ray ─────────────────────────────────────────────────────────
# Limit the object store to 2 GB to avoid reserving too much system RAM upfront.
if ray.is_initialized():
    ray.shutdown()
ray.init(object_store_memory=2 * 1024 ** 3)

# ── 2. Load & encode data, then push to shared memory ─────────────────────────
dataset  = pd.read_csv("../2.Data_Preparation/train_bench.tsv", sep="\t")
all_data = create_one_hot_sets(dataset)

# ray.put() stores the array in shared memory and returns a lightweight reference.
# Workers receive this reference, not a copy — prevents "object too large" errors.
data_ref = ray.put(all_data)
del all_data   # remove local reference so Python can garbage-collect the RAM
gc.collect()
print("Dataset loaded into Ray object store.")

# ── 3. Define search space ─────────────────────────────────────────────────────
search_space = {
    "num_layers":       tune.choice([2, 3, 4, 5]),
    "hidden_sizes":     tune.sample_from(_generate_hidden_sizes),
    "dropout":          tune.uniform(0.1, 0.5),
    "lr":               tune.loguniform(1e-4, 1e-3),   # log scale captures wide range
    "batch_size":       tune.choice([10, 20]),
    "num_lstm_layers":  tune.choice([1, 2]),
    "lstm_hidden_size": tune.choice([64, 128]),
    "data_ref":         data_ref,    # lightweight reference, not the data itself
}

# ── 4. Launch HPO ─────────────────────────────────────────────────────────────
# num_samples=15: test 15 random configurations.
# resources_per_trial: 1 GPU per trial ensures sequential execution on single-GPU machines.
result = tune.run(
    _tune_trainable,
    config=search_space,
    num_samples=15,
    resources_per_trial={"cpu": 1, "gpu": 1 if torch.cuda.is_available() else 0},
)

# ── 5. Report best result ──────────────────────────────────────────────────────
best_trial = result.get_best_trial("mcc", "max", "last")
print("Best config :", best_trial.config)
print("Best CV MCC :", best_trial.last_result["mcc"])

/Users/negin/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-18 16:11:32,382	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-18 16:11:32,704	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-18 16:11:40,585	INFO worker.py:2013 -- Started a local Ray instance.
/Users/negin/miniconda3/lib/python3.13/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON

Dataset loaded into Ray object store.


(_tune_trainable pid=6270) /var/folders/8v/dft5tkg533s0f0n7c3jl_mwr0000gn/T/ipykernel_6230/1673903641.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
(_tune_trainable pid=6271) /var/folders/8v/dft5tkg533s0f0n7c3jl_mwr0000gn/T/ipykernel_6230/1673903641.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the re

Trial name,hidden_layers_size,loss,mcc
_tune_trainable_c5d2b_00000,"[64, 1024, 32, 512, 128]",-0.271236,0.271236
_tune_trainable_c5d2b_00001,"[32, 128, 512, 1024, 256]",-0.056249,0.056249
_tune_trainable_c5d2b_00002,"[32, 128]",-0.655584,0.655584
_tune_trainable_c5d2b_00003,"[256, 512, 1024, 128, 64]",-0.894997,0.894997
_tune_trainable_c5d2b_00004,"[128, 256, 64, 32]",-0.755137,0.755137
_tune_trainable_c5d2b_00005,"[1024, 512, 128, 64, 32]",-0.888805,0.888805
_tune_trainable_c5d2b_00006,"[256, 64]",-0.726778,0.726778
_tune_trainable_c5d2b_00007,"[1024, 256, 64, 32]",-0.0469642,0.0469642
_tune_trainable_c5d2b_00008,"[1024, 256, 64, 32]",-0.362946,0.362946
_tune_trainable_c5d2b_00009,"[1024, 512, 256, 32]",-0.166451,0.166451


(_tune_trainable pid=7402) /var/folders/8v/dft5tkg533s0f0n7c3jl_mwr0000gn/T/ipykernel_6230/1673903641.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.) [repeated 2x across cluster]
(_tune_trainable pid=7761) /var/folders/8v/dft5tkg533s0f0n7c3jl_mwr0000gn/T/ipykernel_6230/1673903641.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning 

Best config : {'num_layers': 5, 'hidden_sizes': [256, 512, 1024, 128, 64], 'dropout': 0.32710099811576265, 'lr': 0.0001842122766057993, 'batch_size': 20, 'num_lstm_layers': 2, 'lstm_hidden_size': 128, 'data_ref': ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000001e1f505)}
Best CV MCC : 0.8949973295326205
